# Hyperparameter Tuning for Hidden Layers, Nodes, and Optimizer using KerasTuner

## Introduction

When building an Artificial Neural Network (ANN), one of the biggest questions is:

- How many hidden layers should I use?
- How many neurons should each hidden layer have?
- Which optimizer should I choose?

There is **no fixed answer** because every dataset is different.

Instead of manually trying hundreds of combinations, **KerasTuner** automatically searches for the best architecture and training configuration.

---

# Why Do We Need Hyperparameter Tuning?

Suppose we build three models.

## Model A

```
Hidden Layers = 1

Nodes = 32

Optimizer = SGD
```

Accuracy = **87%**

---

## Model B

```
Hidden Layers = 2

Nodes = 64

Optimizer = Adam
```

Accuracy = **94%**

---

## Model C

```
Hidden Layers = 4

Nodes = 256

Optimizer = RMSProp
```

Accuracy = **90%**

Which model is best?

Instead of manually testing every combination, KerasTuner automates this process.

---

# What Can KerasTuner Tune?

```
ANN

├── Number of Hidden Layers
├── Number of Neurons
├── Optimizer
├── Learning Rate
├── Activation Function
├── Dropout Rate
├── Batch Size
└── Regularization
```

---

# Hyperparameter 1: Number of Hidden Layers

## Why Do We Tune It?

Hidden layers determine **how deep** the neural network is.

### Too Few Hidden Layers

```
Input
 ↓
Hidden
 ↓
Output
```

Problems

- Underfitting
- Cannot learn complex patterns
- Low accuracy

---

### Too Many Hidden Layers

```
Input
 ↓
Hidden
 ↓
Hidden
 ↓
Hidden
 ↓
Hidden
 ↓
Output
```

Problems

- Overfitting
- Slower training
- More computation
- Harder optimization

---

### Goal

Find the **smallest architecture** that achieves strong validation performance.

---

# Hyperparameter 2: Number of Nodes (Neurons)

## Why Do We Tune It?

Neurons control the **capacity** of each hidden layer.

### Too Few Nodes

```
● ●
```

Problems

- Low capacity
- Underfitting

---

### Too Many Nodes

```
● ● ● ● ● ● ● ● ● ● ● ●
```

Problems

- Overfitting
- Large memory usage
- Longer training

---

### Goal

Choose enough neurons to learn meaningful patterns without memorizing the training data.

---

# Hyperparameter 3: Optimizer

## Why Do We Tune It?

Different optimizers update weights differently.

Different datasets respond differently to each optimizer.

Common optimizers

| Optimizer | Characteristics |
|------------|----------------|
| SGD | Simple, slower convergence, often good generalization |
| RMSProp | Adaptive learning rate, works well for sequential data |
| Adam | Momentum + RMSProp, fast convergence |
| AdamW | Adam with proper weight decay, preferred for many modern deep learning models |

---

# Why Different Optimizers?

Imagine climbing down a mountain.

### SGD

```
↓

↓

↓

↓

```

Small steps.

Slow but steady.

---

### RMSProp

```
↓

↓↓

↓

↓↓↓

```

Adjusts step size according to recent gradients.

---

### Adam

```
↓

↓↓↓

↓↓↓

↓↓↓↓
```

Uses:

- Momentum
- Adaptive learning rate

Usually reaches the minimum faster.

---

# Complete Hyperparameter Search Space

Example

```python
Hidden Layers

1
2
3
4
```

```
Neurons

32
64
128
256
```

```
Optimizer

adam

sgd

rmsprop
```

Total combinations

```
4 × 4 × 3

=

48 Different Models
```

Manually testing all of them is inefficient.

---

# KerasTuner Workflow

```
Define Search Space
          ↓
Build Dynamic Model
          ↓
Random Search / Hyperband / Bayesian
          ↓
Train Multiple Models
          ↓
Compare Validation Metrics
          ↓
Choose Best Hyperparameters
          ↓
Train Final Model
```

---

# Complete Code Example

## Step 1: Install KerasTuner

```bash
pip install keras-tuner
```

---

## Step 2: Import Libraries

```python
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
```

---

## Step 3: Create the Model Building Function

```python
def build_model(hp):

    model = keras.Sequential()

    # Tune number of hidden layers
    for i in range(
        hp.Int(
            "hidden_layers",
            min_value=1,
            max_value=4,
            step=1
        )
    ):

        # Tune neurons in each layer
        model.add(
            keras.layers.Dense(
                units=hp.Choice(
                    f"units_{i}",
                    values=[32,64,128,256]
                ),
                activation="relu"
            )
        )

    model.add(
        keras.layers.Dense(
            1,
            activation="sigmoid"
        )
    )

    # Tune optimizer
    optimizer = hp.Choice(
        "optimizer",
        values=[
            "adam",
            "sgd",
            "rmsprop"
        ]
    )

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model
```

---

## Why This Code?

### `hp.Int()`

```python
hp.Int(
    "hidden_layers",
    1,
    4
)
```

Allows KerasTuner to test:

```
1 Layer

2 Layers

3 Layers

4 Layers
```

---

### `hp.Choice()`

```python
hp.Choice(
    "units",
    [32,64,128,256]
)
```

Allows KerasTuner to test:

```
32

64

128

256
```

neurons.

---

### Optimizer Choice

```python
hp.Choice(
    "optimizer",
    [
        "adam",
        "sgd",
        "rmsprop"
    ]
)
```

Tests all three optimizers automatically.

---

# Step 4: Create the Tuner

```python
tuner = kt.RandomSearch(

    build_model,

    objective="val_accuracy",

    max_trials=10,

    directory="my_dir",

    project_name="ann_tuning"

)
```

---

## Why RandomSearch?

RandomSearch

- Easy to understand
- Good baseline
- Faster than exhaustive search

---

# Step 5: Start Searching

```python
tuner.search(

    X_train,

    y_train,

    validation_data=(X_test, y_test),

    epochs=20

)
```

KerasTuner now:

- Builds different ANN architectures
- Changes hidden layers
- Changes neurons
- Changes optimizer
- Evaluates validation accuracy

---

# Step 6: Get Best Hyperparameters

```python
best_hp = tuner.get_best_hyperparameters()[0]
```

---

# Step 7: Print Best Values

```python
print(best_hp.get("hidden_layers"))

print(best_hp.get("units_0"))

print(best_hp.get("optimizer"))
```

Example Output

```
Hidden Layers

3

Units

128

Optimizer

adam
```

---

# Step 8: Build the Best Model

```python
model = tuner.hypermodel.build(best_hp)
```

Train it normally.

```python
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=50
)
```

---

# Why Use Validation Accuracy?

Training accuracy measures performance on data the model has already seen.

Validation accuracy measures performance on unseen data.

A model with:

```
Training Accuracy = 99%

Validation Accuracy = 82%
```

is likely overfitting.

Always use **validation metrics** when selecting hyperparameters.

---

# Best Practices

- Scale numerical features before training.
- Use EarlyStopping during tuning.
- Start with a small search space.
- Tune the most important hyperparameters first.
- Save the best model after tuning.
- Retrain the final model using the selected hyperparameters.

---

# Common Mistakes

❌ Using training accuracy as the tuning objective.

❌ Searching over an excessively large hyperparameter space.

❌ Ignoring validation loss.

❌ Training every model for too many epochs.

❌ Not scaling input features before training.

---

# Interview Points

- Hidden layers determine model depth.
- Neurons determine the learning capacity of each layer.
- Optimizers control how weights are updated.
- KerasTuner automates hyperparameter search.
- `hp.Int()` defines an integer search range.
- `hp.Choice()` defines a list of candidate values.
- Validation metrics should guide hyperparameter selection.

---

# Key Takeaways

- Hidden layers, neurons, and optimizers are among the most influential ANN hyperparameters.
- KerasTuner automates architecture search and optimizer selection.
- `hp.Int()` is commonly used to tune the number of hidden layers.
- `hp.Choice()` is used for categorical options such as neurons and optimizers.
- The best model is the one that generalizes well on validation data, not necessarily the one with the highest training accuracy.

---

# 2. Interview Questions & Answers

## Beginner Questions

### 1. Why do we tune the number of hidden layers?

**Answer:**
To find the right model depth. Too few hidden layers can underfit the data, while too many may increase the risk of overfitting and computational cost.

---

### 2. Why do we tune the number of neurons?

**Answer:**
The number of neurons controls the capacity of each layer. Choosing an appropriate number helps the model learn meaningful patterns without unnecessary complexity.

---

### 3. Why do we tune the optimizer?

**Answer:**
Different optimizers update model weights differently. The most effective optimizer depends on the dataset and problem, so tuning helps identify the best choice.

---

### 4. What is `hp.Int()`?

**Answer:**
`hp.Int()` defines an integer-valued hyperparameter search space, such as the number of hidden layers.

---

### 5. What is `hp.Choice()`?

**Answer:**
`hp.Choice()` lets KerasTuner choose from a predefined list of values, such as neuron counts or optimizer names.

---

## Intermediate Questions

### 6. Why is `objective="val_accuracy"` commonly used?

**Answer:**
Validation accuracy measures how well the model performs on unseen data, making it a better indicator of generalization than training accuracy.

---

### 7. Why use a loop to create hidden layers?

**Answer:**
A loop allows the architecture to be built dynamically based on the number of hidden layers selected by KerasTuner.

---

### 8. Why not search hundreds of hyperparameter combinations?

**Answer:**
A very large search space significantly increases training time and computational cost. Start with a focused search space and expand only if needed.

---

### 9. Why is feature scaling important before tuning?

**Answer:**
Feature scaling helps neural networks converge faster and makes optimizer comparisons more reliable.

---

### 10. Which optimizer should you try first?

**Answer:**
Adam is a strong default choice for many deep learning tasks because it combines adaptive learning rates with momentum. However, the best optimizer should be selected based on validation performance.

---

## Scenario-Based Questions

### Q1. KerasTuner chooses 2 hidden layers instead of your manually designed 5-layer model. Which one should you use?

**Answer:**
Prefer the model with better validation performance. A simpler architecture that generalizes well is usually better than a more complex one with higher training accuracy.

---

### Q2. Your model trains very slowly during tuning. What changes would you make?

**Answer:**

* Reduce the search space
* Use EarlyStopping
* Reduce the number of epochs
* Use Hyperband instead of Random Search for larger search spaces

---

### Q3. KerasTuner selects SGD, but Adam gives slightly higher training accuracy. Which optimizer would you trust?

**Answer:**
Trust the optimizer that achieves the best validation performance, since it is more likely to generalize to unseen data.

---

# 3. 30-Second Revision

* Tune **hidden layers** to control model depth.
* Tune **neurons** to control model capacity.
* Tune the **optimizer** to improve convergence.
* `hp.Int()` tunes integer values (e.g., hidden layers).
* `hp.Choice()` tunes categorical options (e.g., optimizer, neurons).
* Select hyperparameters using **validation metrics**, not training accuracy.

---

# 4. 2-Minute Revision

* Hidden layers determine how deeply an ANN can learn.
* Neurons determine how much information each layer can represent.
* Optimizers such as SGD, RMSProp, and Adam update weights differently.
* KerasTuner automates searching across combinations of these hyperparameters.
* Build a dynamic model using `hp.Int()` and `hp.Choice()`, evaluate using validation metrics, and retrain the final model with the best configuration.

---

